In [ ]:
#Turns out, you have to use python 3.10 in order for this all to work. The older femr model must also be used
!sudo apt-get update -y
!sudo apt-get install -y python3.10 python3.10-venv
!python3.10 -m venv /content/femr_venv
!/content/femr_venv/bin/pip install "femr[models]==0.1.16"

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3.10 is already the newest version (3.10.12-1~22.04.15).
python3.10-venv is already the 

In [ ]:
#Libraries on which femr work depends
!/content/femr_venv/bin/pip install "jax[cpu]==0.4.8" "jaxlib==0.4.7" \
  -f https://storage.googleapis.com/jax-releases/jax_releases.html
!/content/femr_venv/bin/pip install "scipy<1.11"

Looking in links: https://storage.googleapis.com/jax-releases/jax_releases.html
DEPRECATION: The HTML index page being used (https://storage.googleapis.com/jax-releases/jax_releases.html) is not a proper HTML 5 document. This is in violation of PEP 503 which requires these pages to be well-formed HTML 5 documents. Please reach out to the owners of this index page, and ask them to update this index page to a valid HTML 5 document. pip 22.2 will enforce this behaviour change. Discussion can be found at https://github.com/pypa/pip/issues/10825
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.1/66.1 MB 8.0 MB/s eta 0:00:00
  Attempting uninstall: jaxlib
    Found existing installation: jaxlib 0.6.2
    Uninstalling jaxlib-0.6.2:
      Successfully uninstalled jaxlib-0.6.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 16.2 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.3
    Uninstalling scipy-1.15.3:
      Successfully uninstalled sc

In [ ]:
!/content/femr_venv/bin/python -c "import jax.numpy as jnp; jnp.DeviceArray = jnp.ndarray; import femr.models.transformer"

In [ ]:
#Here you paste your token for access to the MOTOR t-base model.
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
#This downloads the model into this colab environment
from huggingface_hub import snapshot_download
motor_path = snapshot_download("StanfordShahLab/motor-t-base")
print(motor_path)

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/models--StanfordShahLab--motor-t-base/snapshots/71f506ce2f5de7e9291e990d02cf0b0ef56139dc


In [ ]:
#This is the code to make simulated patients. The patient information gets stored in "simple_femr_input/simulated_patients.csv."
%%writefile make_patients.py
"""
Here I have generated 2 SIMULATED patients in the "simple femr" CSV format, which the
`etl_simple_femr` tool turns into a femr PatientDatabase.

This mirrors the official FEMR generator
(tutorials/synthetic_data_generation/generate_simple_femr.py) and the schema
enforced by femr.etl_pipelines.simple, so the output is byte-compatible with the
real ETL.

KEY RULES (enforced by femr.etl_pipelines.simple.convert_file_to_event_file):
  * Header MUST be: patient_id,start,code,value,dosage,visit_ids,lab_units,clarity_source
      - patient_id, start, code, value are special.
      - Every OTHER column (dosage, visit_ids, lab_units, clarity_source) is
        attached to the Event as metadata. Empty string -> None.
  * `start` must be ISO format (YYYY-MM-DD or full ISO datetime).
  * Events do NOT need to be pre-sorted; the ETL sorts by (patient_id, start).
"""
import csv
import datetime
import os

OUT_DIR = "simple_femr_input"          # folder of CSV(s) you'll feed to etl_simple_femr
OUT_FILE = "simulated_patients.csv"

DEMOGRAPHIC_CODES = {
    "birth":  "Birth/Birth",
}

#For convenience, you can reference this dict when making patient events below like so: VITALS_CODES["systolic_bp"]
VITALS_CODES = {
    "systolic_bp":      "LOINC/8480-6",    # mmHg
    "diastolic_bp":     "LOINC/8462-4",    # mmHg
    "heart_rate":       "LOINC/8867-4",    # beats/min
    "respiratory_rate": "LOINC/9279-1",    # breaths/min
    "body_temperature": "LOINC/8310-5",    # Cel (or [degF])
    "spo2":             "LOINC/2708-6",    # %   (use LOINC/59408-5 for the pulse-ox-specific concept)
    "body_height":      "LOINC/8302-2",    # cm  (or [in_us])
    "body_weight":      "LOINC/29463-7",   # kg  (or [lb_av])
    "bmi":              "LOINC/39156-5",   # kg/m2
}


def write_patient(writer, patient_id, birth_date, gender_code, race_code, timeline):
    writer.writerow([patient_id, birth_date.isoformat(), DEMOGRAPHIC_CODES["birth"],
                     "", "", 1, "", "PATIENT"])
    writer.writerow([patient_id, birth_date.isoformat(), gender_code,
                     "", "", 1, "", "PATIENT"])
    writer.writerow([patient_id, birth_date.isoformat(), race_code,
                     "", "", 1, "", "PATIENT"])
    for e in timeline:                              # This is the backbone code that generates clinical events
        writer.writerow([
            patient_id, e["date"].isoformat(), e["code"],
            e.get("value", ""), e.get("dosage", ""),
            e.get("visit_ids", 1), e.get("lab_units", ""), e.get("clarity_source", ""),
        ])


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    out_path = os.path.join(OUT_DIR, OUT_FILE)

    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(
            ["patient_id", "start", "code", "value", "dosage", "visit_ids", "lab_units", "clarity_source"]
        )

# >>> Here is where we make the simulated patients. Codes are ones you confirmed is in the model dictionary via inspect_motor_dictionary.py.    <

        # This is a 67F patient with a BMI of 50, a previous stroke, and an incarcerated hernia with a previous failed repair
        d = datetime.date
        write_patient(writer, patient_id=900002, birth_date=d(1959, 3, 14),
                      gender_code="Gender/F", race_code="Race/3",   # Female, Black or African American
            timeline=[
                {"code": VITALS_CODES["systolic_bp"],   "date": d(2020, 7, 9), "value": 160.0, "lab_units": "mmHg", "clarity_source": "LAB_RESULT", "visit_ids": 9},
                {"code": VITALS_CODES["diastolic_bp"],   "date": d(2020, 7, 9), "value": 95.0,  "lab_units": "mmHg", "clarity_source": "LAB_RESULT", "visit_ids": 9},
                {"code": "SNOMED/230690007", "date": d(2010, 2, 1), "clarity_source": "DIAGNOSIS", "visit_ids": 1},     #prior stroke
                {"code": "LOINC/4548-4", "date": d(2020, 8, 9), "value": 10.0,   "lab_units": "%",    "clarity_source": "LAB_RESULT", "visit_ids": 9},     #HgA1C
                {"code": VITALS_CODES["bmi"], "date": d(2019, 2, 1), "value": 50.0,  "lab_units": "kg/m2", "clarity_source": "LAB_RESULT", "visit_ids": 9},
                {"code": "CPT4/49652", "date": d(2015, 9, 22), "clarity_source": "PROCEDURES", "visit_ids": 2},    #prior repair
                {"code": "CPT4/49653", "date": d(2022, 6, 5), "clarity_source": "PROCEDURES", "visit_ids": 5},     #current repair
            ],
        )

        # This is a 75M patient with acute cholecystitis with a past surgical history of splenectomy
        write_patient(writer, patient_id=900001, birth_date=d(1950, 1, 7),
                      gender_code="Gender/M", race_code="Race/5",   # Male, White
            timeline=[
                {"code": "SNOMED/59771005", "date": d(2021, 9, 22), "clarity_source": "DIAGNOSIS", "visit_ids": 17}, #acute cholecystitis
                {"code": VITALS_CODES["systolic_bp"], "date": d(2019, 2, 1), "value": 145.0, "lab_units": "mmHg", "clarity_source": "LAB_RESULT", "visit_ids": 10},
                {"code": VITALS_CODES["diastolic_bp"], "date": d(2019, 2, 1), "value": 90.0,  "lab_units": "mmHg", "clarity_source": "LAB_RESULT", "visit_ids": 10},
                {"code": "RxNorm/310793",      "date": d(2019, 3, 10), "dosage": 500, "lab_units": "mg", "clarity_source": "MED_ORDER", "visit_ids": 13}, #HCTZ/irbesartan
                {"code": "CPT4/47562", "date": d(2001, 7, 22), "clarity_source": "PROCEDURES", "visit_ids": 8},     #prior splenectomy
                {"code": "CPT4/47562", "date": d(2021, 9, 22), "clarity_source": "PROCEDURES", "visit_ids": 17},     #lap chole
            ],
        )

if __name__ == "__main__":
    main()

Overwriting make_patients.py


In [ ]:
#Do this next
%%bash
python make_patients.py
# -> simple_femr_input/simulated_patients.csv

In [ ]:
#You have to point these commands to the femr environment (femr_venv) you've created using python 3.10 and femr 0.1.16
#etl_simple_femr here takes a folder of CSVs (in this case, the folder is simple_femr_input) and makes "my_extract"
!/content/femr_venv/bin/etl_simple_femr simple_femr_input my_extract /tmp/femr_simple_tmp --num_threads 1

2026-06-27 02:20:29,922 [MainThread  ] [INFO ]  Extracting from OMOP with arguments Namespace(simple_source='simple_femr_input', target_location='/content/my_extract', temp_location='/tmp/femr_simple_tmp', num_threads=1, athena_download=None)
2026-06-27 02:20:29,922 [MainThread  ] [INFO ]  Already converted to events, skipping
2026-06-27 02:20:29,922 [MainThread  ] [INFO ]  Already converted to patients, skipping
2026-06-27 02:20:29,922 [MainThread  ] [INFO ]  Already converted to extract, skipping


In [ ]:
#This takes my_extract and makes it into a femr.datasets.PatientDatabase, something the model can use
#I think I have to do this before computing representations; this makes our prediction times?
#The rule of thumb is just that anything touching femr goes through /content/femr_venv/bin/
%%writefile make_pred_times.py
import csv, femr.datasets
db = femr.datasets.PatientDatabase("my_extract")
with open("prediction_times.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["patient_id", "prediction_time"])
    for pid in db:
        last = max(e.start for e in db[pid].events)
        w.writerow([pid, last.isoformat()])
print("wrote prediction_times.csv")



Overwriting make_pred_times.py


In [ ]:
!/content/femr_venv/bin/python make_pred_times.py

wrote prediction_times.csv


In [ ]:
#You will be using this to compute the representations; again, you're pointing it to your femr environment
#You have to use this PATH= thing because femr_compute_representations shells out to another script, and it can't
#find venv's bin/ on PATH for that child process otherwise.
!PATH="/content/femr_venv/bin:$PATH" /content/femr_venv/bin/femr_compute_representations \
  --data_path my_extract \
  --model_path /root/.cache/huggingface/hub/models--StanfordShahLab--motor-t-base/snapshots/71f506ce2f5de7e9291e990d02cf0b0ef56139dc \
  --prediction_times_path prediction_times.csv \
  --batch_size 1024 \
  motor_reprs.pkl

2026-06-27 02:20:40,590 [MainThread  ] [INFO ]  Preparing batches with Namespace(directory='/tmp/tmpdh8ufezd/task_batches', data_path='my_extract', dictionary_path='/root/.cache/huggingface/hub/models--StanfordShahLab--motor-t-base/snapshots/71f506ce2f5de7e9291e990d02cf0b0ef56139dc/dictionary', task='labeled_patients', transformer_vocab_size=65536, clmbr_survival_dictionary_path=None, labeled_patients_path='prediction_times.csv', is_hierarchical=True, seed=97, val_start=70, test_start=85, batch_size=1024, note_embedding_data=None, limit_to_patients_file=None, limit_before_date=None, num_clmbr_tasks=8192)
2026-06-27 02:20:46,047 [MainThread  ] [INFO ]  Wrote config ...
2026-06-27 02:20:46,048 [MainThread  ] [INFO ]  Starting to load
When mapping codes, dropped 14174 out of 40996
When mapping codes, dropped 14174 out of 40996
2026-06-27 02:21:17,029 [MainThread  ] [INFO ]  Loaded
When mapping codes, dropped 14174 out of 40996
2026-06-27 02:21:31,099 [MainThread  ] [INFO ]  Number of trai

In [ ]:
#This is one way you could inspect output
import pickle
r = pickle.load(open("motor_reprs.pkl", "rb"))
print(r.keys())
print("representations", r["representations"].shape)   # (2, repr_dim)
print("patient_ids", r["patient_ids"])                 # 900001, 900002
print("prediction_times", r["prediction_times"])

dict_keys(['representations', 'patient_ids', 'prediction_times'])
representations (2, 769)
patient_ids [900001 900002]
prediction_times [datetime.datetime(2021, 9, 22, 0, 0) datetime.datetime(2022, 6, 5, 0, 0)]


In [ ]:
#This makes a file that we can subsequently use to further check the validity of the representations
%%writefile check_representations.py
"""
check_representations.py
------------------------
Checking motor_reprs.pkl from femr_compute_representations to decide whether
the embeddings are real or degenerate.

This code checks for the following warning signs of degenerate embeddings:
  * NaNs / infinities
  * all-zero or near-zero rows (no signal)
  * dimensions that are constant across all rows (carry no information)
  * two patients whose embeddings are essentially identical (cosine ~ 1.0)
    even though their timelines differ
"""

import sys
import pickle
import numpy as np


def main():
    path = sys.argv[1] if len(sys.argv) > 1 else "motor_reprs.pkl"
    with open(path, "rb") as f:
        r = pickle.load(f)

    print("keys:", list(r.keys()))
    reps = np.asarray(r["representations"], dtype=float)
    pids = np.asarray(r["patient_ids"])
    times = r.get("prediction_times")
    n, dim = reps.shape
    print(f"representations shape: {reps.shape}  ({n} prediction times, dim={dim})")
    print("patient_ids:", pids.tolist())
    if times is not None:
        print("prediction_times:", list(times)[:n])
    print("-" * 70)

    issues = []

    # 1. NaN / inf
    n_nan = int(np.isnan(reps).sum())
    n_inf = int(np.isinf(reps).sum())
    print(f"NaNs: {n_nan}   infs: {n_inf}")
    if n_nan or n_inf:
        issues.append("contains NaN/inf")

    # 2. per-row norms (exclude a possible constant bias column at the end)
    norms = np.linalg.norm(reps, axis=1)
    print("per-row L2 norm:", np.round(norms, 4).tolist())
    if np.any(norms < 1e-6):
        issues.append("one or more rows are all-zero")

    # 3. constant dimensions (zero variance across rows)
    col_var = reps.var(axis=0)
    n_const = int((col_var < 1e-12).sum())
    print(f"constant dimensions (zero variance across rows): {n_const} / {dim}")
    # note: 1 constant dim is normal -- MOTOR appends a constant bias/intercept term
    if n_const > max(1, dim // 2):
        issues.append(f"{n_const}/{dim} dims are constant -- little information")

    # 4. distinctness between rows (cosine similarity)
    if n >= 2:
        # cosine similarity matrix
        unit = reps / np.clip(norms[:, None], 1e-12, None)
        cos = unit @ unit.T
        print("-" * 70)
        print("pairwise cosine similarity:")
        np.set_printoptions(precision=4, suppress=True)
        print(cos)
        # flag near-identical distinct rows
        iu = np.triu_indices(n, k=1)
        max_offdiag = cos[iu].max() if iu[0].size else 0.0
        print(f"max off-diagonal cosine (closest two rows): {max_offdiag:.4f}")
        if max_offdiag > 0.999:
            issues.append("two rows are nearly identical (cosine > 0.999) -- "
                          "patients may not be differentiated")
    print("-" * 70)

    # verdict
    if issues:
        print("POTENTIALLY DEGENERATE:")
        for i in issues:
            print("  -", i)
    else:
        print("LOOKS HEALTHY: non-zero norms, varied dimensions, and rows are "
              "distinguishable from each other.")
        print("(With only a couple of patients this is a smoke test, not proof of "
              "clinical validity -- but it rules out the obvious failures.)")


if __name__ == "__main__":
    main()

Overwriting check_representations.py


In [ ]:
#What to look for in its output:
#per-row L2 norm — should be clearly non-zero (a row of ~0 means no signal).
#constant dimensions — 1 is normal (MOTOR appends a constant bias term); a large fraction being constant is a red flag.
#pairwise cosine similarity — your two patients have different timelines, so the off-diagonal should be well below 1.0.
#If it's ~0.999, the model isn't distinguishing them.
#Final line gives a plain verdict.

!python check_representations.py motor_reprs.pkl

keys: ['representations', 'patient_ids', 'prediction_times']
representations shape: (2, 769)  (2 prediction times, dim=769)
patient_ids: [900001, 900002]
prediction_times: [datetime.datetime(2021, 9, 22, 0, 0), datetime.datetime(2022, 6, 5, 0, 0)]
----------------------------------------------------------------------
NaNs: 0   infs: 0
per-row L2 norm: [28.5789, 28.8548]
constant dimensions (zero variance across rows): 1 / 769
----------------------------------------------------------------------
pairwise cosine similarity:
[[1.     0.3441]
 [0.3441 1.    ]]
max off-diagonal cosine (closest two rows): 0.3441
----------------------------------------------------------------------
LOOKS HEALTHY: non-zero norms, varied dimensions, and rows are distinguishable from each other.
(With only a couple of patients this is a smoke test, not proof of clinical validity -- but it rules out the obvious failure modes.)
